## Using ResNet (Transfer Learning) Step by Step

Instead of training from scratch, we’ll use a pretrained model:

### What is ResNet?

ResNet = Residual Network

It was introduced in:

Microsoft Research

Paper: “Deep Residual Learning for Image Recognition”

Authors include Kaiming He

Popular model:

ResNet (we'll use ResNet18)

### Why ResNet is Special 

In normal deep networks:

H(x) ,is what the layer tries to learn.

In ResNet, the layer learns:
F(x)=H(x)−x

So the output becomes:

H(x)=F(x)+x

This is called a skip connection or residual connection.

Instead of learning full mapping, it learns the difference.

Why is this powerful?

Because deep networks suffer from:

Vanishing gradients which occurs when the gradient is small, it causes problem in backpropagation.

Degradation problem (accuracy gets worse when network gets deeper)

Residual connections help gradients flow directly backward.

## ResNet18 Transfer Learning

### STEP 1 — Imports

In [83]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os

### STEP 2 — Device

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### STEP 3 — Dataset Path

In [85]:
path = "C:/Users/OLIVE/.cache/kagglehub/datasets/mayank07thakur/happy-vs-sad-people-cnn-classification/versions/1"


### STEP 4 — Transforms
IMPORTANT: ResNet needs ImageNet normalization

In [86]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),          # ResNet default input size (imagenet models are typically trained on 224x224 images, so we resize our input images to this size to ensure compatibility with the pre-trained model)
    transforms.ToTensor(),
    transforms.Normalize(                   # Normalize images using ImageNet mean and std, which is important when using pre-trained models like ResNet that were trained on ImageNet; This ensures that the input images are normalized in the same way as the images used to train the model, which can help improve performance
        mean=[0.485, 0.456, 0.406],         # ImageNet mean
        std=[0.229, 0.224, 0.225]           # ImageNet std
    )                                       
])

# What is Normalization?
# Normalization is the process of transforming data to have a specific statistical distribution - typically with a mean of 0 and standard deviation of 1.



#### 1. WHAT THIS TRANSFORM DOES - STEP BY STEP
```
Step 1: Resize((224, 224))
Purpose: Make all images the size ResNet expects
Why 224? ResNet was designed and trained with 224×224 images

Step 2: ToTensor()
Purpose: Convert PIL Image to PyTorch tensor + scale to [0,1]

Before ToTensor (PIL Image):
- Format: PIL Image object
- Pixel values: 0-255 (integers)
- Shape: (224, 224, 3)  # Height, Width, Channels

After ToTensor (PyTorch Tensor):
- Format: torch.Tensor
- Pixel values: 0.0-1.0 (floats)
- Shape: (3, 224, 224)  # Channels, Height, Width
- Formula: tensor = pixel_value / 255.0

Step 3: Normalize(mean, std)
Purpose: Standardize pixel values to match ImageNet distribution
Formula: output = (input - mean) / std
For each channel independently:
    red_channel = (red_channel - 0.485) / 0.229
    green_channel = (green_channel - 0.456) / 0.224
    blue_channel = (blue_channel - 0.406) / 0.225
```

### STEP 5 — Load Dataset

In [87]:
full_dataset = datasets.ImageFolder(root=path, transform=transform)

print("Classes:", full_dataset.classes)
print("Total images:", len(full_dataset))

Classes: ['Happy People', 'Sad People']
Total images: 351


### STEP 6 — Train/Validation Split

In [88]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 280
Validation size: 71


### STEP 7 — DataLoaders

In [89]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

### STEP 8 — Load Pretrained ResNet18

In [90]:
model = models.resnet18(pretrained=True)

### Freeze all layers except last layer 4 for fine tuning

In [91]:
for param in model.parameters():              # freeze all model parameters
    param.requires_grad = False               # freeze all model parameters so that they are not updated during training; This is important when using a pre-trained model for transfer learning, as we typically want to keep the learned features from the pre-trained model and only train the final classification layer on our specific task

for param in model.layer4.parameters():       # unfreeze the parameters of the last convolutional block (layer4) so that they can be updated during training; This allows us to fine-tune the last few layers of the pre-trained model, which can help improve performance on our specific classification task while still keeping the earlier layers frozen to retain the learned features
    param.requires_grad = True


### Replace final layer

In [92]:
num_features = model.fc.in_features             # Get the number of input features to the final fully connected layer of ResNet, which is necessary to replace it with a new layer that has the correct number of input features for our specific classification task
model.fc = nn.Linear(num_features, 2)           # Replace the final fully connected layer of ResNet with a new layer that has 2 output features (for happy and sad classes), with the same number of input features as the original layer; This allows us to use the pre-trained ResNet model for our specific classification task by only training the final layer while keeping the learned features from the pre-trained model intact

### Only final layer trainable

In [93]:
for param in model.fc.parameters():           # unfreeze the parameters of the final fully connected layer so that they can be updated during training; This is necessary because we want to train the final layer on our specific classification task while keeping the rest of the pre-trained model frozen
    param.requires_grad = True

model = model.to(device)                      # Move the model to the appropriate device (GPU if available, otherwise CPU)

### STEP 9 — Loss and Optimizer

In [94]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': model.fc.parameters(), 'lr': 0.001},            # Only optimize the parameters of the final fully connected layer since the rest of the model is frozen
    {'params': model.layer4.parameters(), 'lr': 0.0001}        # same for last conv layer
])
    

### STEP 10 — Training Loop

In [95]:
torch.cuda.empty_cache()             # Clear the GPU cache to free up memory before starting training; This can help prevent out-of-memory errors when training on a GPU, especially if there are other processes using the GPU or if the model and data are large

epochs = 7

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss:.3f} "
          f"Train Accuracy: {train_acc:.2f}%")

Epoch [1/7] Loss: 13.978 Train Accuracy: 81.43%
Epoch [2/7] Loss: 6.563 Train Accuracy: 92.14%
Epoch [3/7] Loss: 1.919 Train Accuracy: 98.21%
Epoch [4/7] Loss: 3.140 Train Accuracy: 96.79%
Epoch [5/7] Loss: 2.291 Train Accuracy: 98.21%
Epoch [6/7] Loss: 2.405 Train Accuracy: 97.14%
Epoch [7/7] Loss: 2.128 Train Accuracy: 98.21%


### STEP 11 — Validation

In [96]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total

print("Validation Accuracy:", val_acc)

Validation Accuracy: 97.1830985915493


### Why ResNet is Better (Conceptually)

Scratch CNN:

Learns from scratch

351 images only

Risk of overfitting

ResNet:

Learned from 1.2M images

Already knows visual patterns

Only adapts to happy vs sad

So:

Scratch CNN → learns everything
ResNet → fine-tunes knowledge

That’s why transfer learning is powerful.

### Why we unfreeze the last few layers why not the first few layers?

Early CNN layers learn general features like edges and textures that are useful for almost all vision tasks. Therefore, we freeze them during transfer learning. The later layers learn task-specific representations, so we unfreeze the last layers (like layer4) and retrain them to adapt the model to our new dataset (Happy vs Sad classification).
